## from word to braids

In [12]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from geometry_mnist.ptemplates import register
register()


DIM = 3


def _sig(x: np.ndarray) -> np.ndarray:
    """ basically a sigmoid to get a smooth transition from y=0 to y=1 over 1x """
    a = 12
    return 1 / (1 + np.exp(-a * (x - 0.5)))

def _hop(x: np.ndarray) -> np.ndarray:
    """ a gaussian bell to hop in z direction for the active crossing braid """
    a = 30
    b = 0.33
    return b * np.exp(-a * (x - 0.5)**2)


def generate_straight(n_points: int) -> np.ndarray:
    """ just the straight line braid within unit square"""
    pts = np.zeros(shape=(n_points, DIM))
    x, y, z = pts.T # just a "named view" on the pts column 
    x[:] = np.linspace(0, 1, n_points, endpoint=False)
    return pts
    
def generate_l2r_active(n_points: int) -> np.ndarray:
    """ the L->R transition with a positive zhop. hop can easily be made to go over or under by sign flip. """
    
    pts = np.zeros(shape=(n_points, DIM))
    x, y, z = pts.T # just a "named view" on the pts column 
    x[:] = np.linspace(0, 1, n_points, endpoint=False)
    y[:] = _sig(x)
    z[:] = _hop(x)
    return pts
    
def generate_r2l_passive(n_points: int) -> np.ndarray:
    """ the R->L transition without zhop. """
    
    pts = np.zeros(shape=(n_points, DIM))
    x, y, z = pts.T # just a "named view" on the pts column 
    x[:] = np.linspace(0, 1, n_points, endpoint=False)
    y[:] = 1 - _sig(x)
    return pts
    
def generate_newslice(n_points: int, n_braids: int) -> np.ndarray:
    """ helper to generate one op-slice for all braids and fill it with dummy straight lines for each braid. """
    
    # the base line element for one braid at y=z=0 with an extra batch braid dimension for broadcasting
    one_line = generate_straight(n_points)[None, :, :]
    
    # a broadcasting matrix to duplicate the basis line n_braid times and increase y coords by 1 each time
    distributor = np.ones(shape=(n_braids, 1, DIM))
    distributor[:, 0, 1] = np.arange(n_braids) + 1
    
    # elementwise multiplication and broadcasting duplicates the line element with an increasing y coord. temporarily setting the initial y to 1 allows the multiplication to take effect
    one_line[:, :, 1] = 1
    bpts = one_line * distributor
    bpts[:, :, 1] -= 1 # set the line elements to begin at y=0 again

    return bpts



# choosable params
word = [1, 1, 1] # word must be 1-indexed to hold sign+braid_id. 
n_pts_per_ops = 50

# determined params
n_braids = np.max(np.abs(word)) + 1
n_ops = len(word) 

# generate the braid multi-pointlist
bpts = np.zeros(shape=(n_braids, n_ops*n_pts_per_ops, 3))

# build word as points
for i, op in enumerate(word):
    
    # extract braid_ids and hop_dir; braid indices everywhere else but the word input must be 0-indexed!
    braid_id_active  = abs(op) - 1 # index of the braid that is targeted by the op
    braid_id_passive = abs(op) -1 + 1 # index of the braid that has to move as well to swap with the active one
    hop_dir = np.sign(op) # whether to hop over or under

    # one fresh "op slice"; all lists just get a straight line
    bpts_newslice = generate_newslice(n_points=n_pts_per_ops, n_braids=n_braids)
    
    # then overwrite the two braids that are part of the op with crossing points!
    pts_active = generate_l2r_active(n_points=n_pts_per_ops) # the active braid going from left to right
    pts_passive = generate_r2l_passive(n_points=n_pts_per_ops) # the passive braid (one after) going from right to left
    # adjust y coordinates of points to braid position (they are all generated in the unit square)
    pts_active[:, 1]  += braid_id_active
    pts_passive[:, 1] += (braid_id_active)
    # finally flip sign of z-coords depending on hop direction (to go under or over the passive braid)
    pts_active[:, 2] *= hop_dir 
    # overwrite the straight lines with the two crossings
    bpts_newslice[braid_id_active, :, :] = pts_active
    bpts_newslice[braid_id_passive, :, :] = pts_passive

    # offset the entire new slice to the current "x end position"
    bpts_newslice[:, :, 0] += i
    
    # write the newslice into the main braid-points array
    bpts[:, i*n_pts_per_ops:(i+1)*n_pts_per_ops, :] = bpts_newslice


# now convert these to proper xyz coords (before it was generated basically in pyr)
R = max(1, n_ops / (2*np.pi))

converted = np.zeros_like(bpts)


converted[:, :, 0] = (R + bpts[:, :, 2]) * np.sin(bpts[:, :, 0]/n_ops * 2*np.pi)
converted[:, :, 1] = bpts[:, :, 1]
converted[:, : ,2] = (R + bpts[:, :, 2]) * np.cos(bpts[:, :, 0]/n_ops * 2*np.pi)


In [13]:
# add the data
data = [
    go.Scatter3d(
        x=b[:, 0],
        y=b[:, 1],
        z=b[:, 2],
    )
    for b in converted
]

# add a KOS
data += [
    go.Scatter3d(
        x=[0, 0.5 if ax=="x" else 0], y=[0, 0.5 if ax=="y" else 0], z=[0, 0.5 if ax=="z" else 0],
        mode="lines",
        line=dict(color=col, width=10),
        showlegend=False,
        hoverinfo="skip",
    )
    for ax, col in zip(["x", "y", "z"], ["red", "green", "blue"])
]

go.Figure(
    data=data,
    layout=dict(
        template="mine_dark+mine_3d",
        scene=dict(
            aspectmode="data",
            camera=dict(
                eye=dict(x=3.0, y=0, z=0.8), # where the camera sits
                center=dict(x=0, y=0, z=0.0), # the point it looks at
                up=dict(x=-1, y=0, z=0), # which direction is "up" on screen
            ),
        ),
        height=800
    ),
)